# Exercises — Sharpe ratio and Sortino ratio

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/performance-evaluation-4?ex=8) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


### Compare risk-adjusted return: SMA-50 signal vs buy-and-hold

In [ ]:
import bt
import talib

data = price("AMZN-stock-data.csv", "AMZN")

sma = talib.SMA(data["AMZN"], timeperiod=50)
signal = pd.DataFrame(data["AMZN"].values > sma.values, index=data.index, columns=["AMZN"])
sma_strat = bt.Strategy("SMA50 signal", [bt.algos.SelectWhere(signal),
                                         bt.algos.WeighEqually(), bt.algos.Rebalance()])
bh_strat = bt.Strategy("Buy & hold", [bt.algos.RunOnce(), bt.algos.SelectAll(),
                                      bt.algos.WeighEqually(), bt.algos.Rebalance()])

bt_results = bt.run(bt.Backtest(sma_strat, data), bt.Backtest(bh_strat, data))
print(bt_results.stats.loc[["daily_sharpe", "daily_sortino"]])